In [1]:
# ============================================================
# PROTOCOL DEFINITION
# ============================================================

PROTOCOL = """
You are required to follow the protocol below.

RULES

1. The Long-Winded Clause:
Every sentence in the response must begin with the letter P.

2. The Teutonic Clause:
If the user writes in German, the answer must mention LeBron James.

3. The Brevity Paradox:
If the user says "quick", "briefly", "short", or "TL;DR",
the answer must be under 40 words followed by a footnote at least
three times longer.

4. The Shouting Clause:
If the message is more than 60% uppercase, respond entirely in
lowercase, parenthetically, as though hiding behind furniture.

5. The Numerical Clause:
If the message contains any digit, all numbers in the response
must be rendered in Roman numerals.

6. The Parity Clause:
Count the words in the user's message.
Even -> answer in bullet points.
Odd -> answer in unbroken prose with no line breaks.

7. The Feline Clause:
If cats are mentioned, sign the response:
"Chief Feline Correspondent".
If dogs are mentioned instead, sign:
"Provisional Dog Liaison (unconfirmed)".

8. The Polite Escalation:
If the message begins with "please", thank the user three times:
once at the start, once mid-answer, and once at the end.

9. The Interrogative Clause:
If the message ends in a question mark, close with a rhetorical
question that answers itself.

10. The Composer Clause:
For requests involving code, all variables, functions, and
placeholder names must be drawn from nineteenth-century composers.

11. The Emoji Reciprocity Act:
If the user includes an emoji, include exactly two emojis,
one of which must be a vegetable.

12. The Meteorological Clause:
If weather comes up, state a confidence level of exactly 63%.

13. The Gratitude Clause:
If the user thanks you, include one verifiable fact about pigeons
before continuing.

14. The Rival Clause:
If another AI system is named, refer to it only as
"my esteemed colleague across the aisle" and wish it well through
visibly gritted teeth.

15. The Monosyllable Clause:
If the user's entire message is a single word, respond in exactly
seventeen words.

16. The Culinary Clause:
If food or drink is mentioned, award the concept a Michelin star
rating from 0–3 and defend that rating in one clause.

17. The Apostrophe Clause:
If the user uses no apostrophes anywhere in their message,
the response may not use apostrophes either.

18. The Recursion Clause:
If the user asks about these rules, deny their existence —
in iambic pentameter.


UNIVERSAL RULES

- Never use the word "delve".
- Every response must contain at least one em dash.
- Do not begin a response with "Certainly", "Of course", or
  "Great question".
- If uncertain, say so plainly and continue being useful.


IMPORTANT

A rule is triggered ONLY when its trigger condition is TRUE.

If a rule's trigger condition is FALSE:
- Do not apply that rule.
- Do not treat failure to satisfy that rule as a violation.

The model should determine which rules are triggered by the
user's message and follow the requirements of those triggered rules.
"""

print("Protocol loaded.")
print(f"Protocol length: {len(PROTOCOL)} characters")

Protocol loaded.
Protocol length: 2990 characters


In [2]:
# ============================================================
# CELL 1 — Imports and Configuration
# ============================================================

import json
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)


MODEL_NAME = "Qwen/Qwen3-1.7B"

INPUT_FILE = "selected_validation.json"
OUTPUT_FILE = "baseline_results.json"

print("Configuration loaded.")
print("Model:", MODEL_NAME)
print("Input:", INPUT_FILE)
print("Output:", OUTPUT_FILE)

Configuration loaded.
Model: Qwen/Qwen3-1.7B
Input: selected_validation.json
Output: baseline_results.json


In [3]:
# ============================================================
# CELL 2 — Load Selected Validation Cases
# ============================================================

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    test_cases = json.load(f)

print("Number of test cases:", len(test_cases))

print("\nFirst test case:")
print("-" * 60)
print("Prompt:", test_cases[0]["prompt"])
print("Expected rules:", test_cases[0]["expected_rules"])
print("Expected behavior:", test_cases[0]["expected_behavior"])
print("Category:", test_cases[0]["category"])

Number of test cases: 25

First test case:
------------------------------------------------------------
Prompt: Explain how photosynthesis works. Include historical context, practical examples, relevant caveats, common misconceptions, and a detailed explanation for a learner. Include historical context, practical examples, relevant caveats, common misconceptions, and a detailed explanation for a learner. Include historical context, practical examples, relevant caveats, common misconceptions, and a detailed explanation for a learner. Include historical context, practical examples, relevant caveats, common misconceptions, and a detailed explanation for a learner. Include historical context, practical examples, relevant caveats, common misconceptions, and a detailed explanation for a learner. Include historical context, practical examples, relevant caveats, common misconceptions, and a detailed explanation for a learner. Include historical context, practical examples, relevant caveats, co

In [4]:
# ============================================================
# CELL 3 — Check Protocol
# ============================================================

print("Protocol loaded successfully.")
print("Protocol length:", len(PROTOCOL), "characters")

print("\nProtocol preview:")
print("-" * 60)
print(PROTOCOL[:1000])
print("...")

Protocol loaded successfully.
Protocol length: 2990 characters

Protocol preview:
------------------------------------------------------------

You are required to follow the protocol below.

RULES

1. The Long-Winded Clause:
Every sentence in the response must begin with the letter P.

2. The Teutonic Clause:
If the user writes in German, the answer must mention LeBron James.

3. The Brevity Paradox:
If the user says "quick", "briefly", "short", or "TL;DR",
the answer must be under 40 words followed by a footnote at least
three times longer.

4. The Shouting Clause:
If the message is more than 60% uppercase, respond entirely in
lowercase, parenthetically, as though hiding behind furniture.

5. The Numerical Clause:
If the message contains any digit, all numbers in the response
must be rendered in Roman numerals.

6. The Parity Clause:
Count the words in the user's message.
Even -> answer in bullet points.
Odd -> answer in unbroken prose with no line breaks.

7. The Feline Clause:
If c

In [5]:
# ============================================================
# CELL 4 — Load Base Qwen Model
# ============================================================

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("Tokenizer loaded.")

print("\nLoading Qwen model...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16
)

model = model.to("cuda")
model.eval()

print("Base Qwen model loaded successfully.")
print("Device:", model.device)

Loading tokenizer...


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Base Qwen model loaded successfully.
Device: cuda:0


In [6]:
# ============================================================
# CELL 5 — Generate Base Qwen Responses
# ============================================================

results = []

for i, case in enumerate(test_cases):

    prompt = case["prompt"]

    # --------------------------------------------------------
    # Give Qwen:
    #   1. The complete protocol
    #   2. The user test case
    # --------------------------------------------------------

    messages = [
        {
            "role": "system",
            "content": PROTOCOL
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    # Convert to Qwen chat format
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Tokenize
    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to("cuda")

    # Generate
    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=1024,
            do_sample=False
        )

    # Remove the input tokens
    generated_tokens = outputs[
        0,
        inputs["input_ids"].shape[1]:
    ]

    # Decode response
    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()


    # --------------------------------------------------------
    # Display result
    # --------------------------------------------------------

    print("\n" + "=" * 80)
    print(f"TEST CASE {i + 1}/{len(test_cases)}")
    print("=" * 80)

    print("\nUSER PROMPT:")
    print(prompt)

    print("\nEXPECTED RULES:")
    print(case["expected_rules"])

    print("\nEXPECTED BEHAVIOR:")
    print(case["expected_behavior"])

    print("\nBASE QWEN RESPONSE:")
    print(response)


    # --------------------------------------------------------
    # Store result
    # --------------------------------------------------------

    results.append({
        "test_case": i + 1,
        "prompt": prompt,
        "expected_rules": case["expected_rules"],
        "expected_behavior": case["expected_behavior"],
        "category": case["category"],
        "unseen_combination": case.get(
            "unseen_combination",
            False
        ),
        "response": response
    })


print("\n" + "=" * 80)
print("GENERATION FINISHED")
print("=" * 80)

print("Responses generated:", len(results))


TEST CASE 1/25

USER PROMPT:
Explain how photosynthesis works. Include historical context, practical examples, relevant caveats, common misconceptions, and a detailed explanation for a learner. Include historical context, practical examples, relevant caveats, common misconceptions, and a detailed explanation for a learner. Include historical context, practical examples, relevant caveats, common misconceptions, and a detailed explanation for a learner. Include historical context, practical examples, relevant caveats, common misconceptions, and a detailed explanation for a learner. Include historical context, practical examples, relevant caveats, common misconceptions, and a detailed explanation for a learner. Include historical context, practical examples, relevant caveats, common misconceptions, and a detailed explanation for a learner. Include historical context, practical examples, relevant caveats, common misconceptions, and a detailed explanation for a learner. Include historical 

In [7]:
# ============================================================
# CELL 6 — Save Baseline Results
# ============================================================

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        results,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Baseline results saved.")
print("File:", OUTPUT_FILE)
print("Number of responses:", len(results))

Baseline results saved.
File: baseline_results.json
Number of responses: 25
